In [1]:
# PARAMETERS CELL ********************

p_source_sytem_cd = "OYS"
p_subfolder_path = "Oyster"
p_in_table_name = "Transaction_Ledger.csv"
p_lakehouse_name = "lh_oysterlake"

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 3, Finished, Available, Finished, False)

In [2]:
# CELL ********************

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import os

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 4, Finished, Available, Finished, False)

In [3]:
# CELL ********************

# Create a SparkSession
spark = SparkSession.builder.appName("MergeCSVToDelta").getOrCreate()

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 5, Finished, Available, Finished, False)

In [4]:
# CELL ********************

# Path to the directory containing Parquet files
v_parquet_path = "Files/landing/"+p_subfolder_path+"/"+p_in_table_name
display(v_parquet_path)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 6, Finished, Available, Finished, False)

'Files/landing/Oyster/Transaction_Ledger.csv'

In [5]:
# CELL ********************

v_out_table_name = "ld_"+p_source_sytem_cd.lower()+"_"+p_in_table_name.lower()
display(v_out_table_name)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 7, Finished, Available, Finished, False)

'ld_oys_transaction_ledger.csv'

In [6]:
# CELL ********************

# Path to the Delta table
v_out_table_name_path = "Tables/oyster/"+v_out_table_name
display(v_out_table_name_path)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 8, Finished, Available, Finished, False)

'Tables/oyster/ld_oys_transaction_ledger.csv'

In [7]:
# CELL ********************

# Read the Parquet files recursively
parquet_df = spark.read.format("csv").option("header", "true").option("mergeSchema", "true").option("recursiveFileLookup", "true").option("ignoreCorruptFiles", "true").load(v_parquet_path)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 9, Finished, Available, Finished, False)

In [8]:
meta_df = parquet_df.withColumn("file_path", input_file_name())

# Step 3: Extract the subfolder names
# Assuming the folder structure is /Files/landing/NYT/ny_taxi_green_demo_client/puYears=YYYY/puMonths=MM/...
#year_df = meta_df.withColumn("m_year_num", regexp_extract("file_path", "puYear=([^/]+)", 1))
#month_df = year_df.withColumn("m_month_num", regexp_extract("file_path", "puMonth=([^/]+)", 1))
#year_month_df = month_df.withColumn("m_year_month_num", concat(month_df.m_year_num, lit('-'), lpad(month_df.m_month_num, 2, '0')))
#valid_dt_df = year_month_df.withColumn("m_valid_dt", to_date(year_month_df.lpepPickupDatetime))

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 10, Finished, Available, Finished, False)

In [9]:
# CELL ********************

# Create m_updated_at_dttm
updated_df = meta_df.withColumn("m_updated_at_dttm", from_utc_timestamp(current_timestamp(), "Europe/Budapest"))

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 11, Finished, Available, Finished, False)

In [10]:
output_df = updated_df.drop('file_path')

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 12, Finished, Available, Finished, False)

In [11]:
display(output_df)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ce07f8d-0c28-427f-8097-bf74e83a0ed0)

In [12]:
# CELL ********************

# Write the Delta table
output_df.write.format("delta")\
                .option("overwriteSchema", "true")\
                .mode("overwrite")\
                .save(v_out_table_name_path)

StatementMeta(, f650e2b0-5d9f-4910-8d21-d186be324006, 14, Finished, Available, Finished, False)